In [1]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import pathlib
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

In [2]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [3]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
dev = tf.config.list_physical_devices()
print('Physical Devices : ', dev)

#tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
dev = tf.config.list_logical_devices()
print('Available Devices : ', dev)

Physical Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Available Devices :  [LogicalDevice(name='/device:CPU:0', device_type='CPU')]


2025-06-15 01:01:40.178042: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# Chapter 11: Deep Learning Text

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

In this section we look at the particular types of deep network architectures that work well when processing textual time series, as
well as other aspects specific to preparing and processing textual input.

## 11.4 The Transformer Architecture

- Starting 2017 a new model architecture started overtaking recurrent neural networks across
  most NLP tasks: The Transformer
- Seminal paper: Ashish Vaswani et al., “Attention is all you need” (2017), https://arxiv.org/abs/1706.03762
- A simple mechanism called **neural attention** could be used to build powerful sequence models
  that didn't feature any recurrent layers or convolution layers.
- Neural attention has fast become one of the most influential ides in deep learning.

### 11.4.1 Understanding Self-attention

Pseudo-code implementation of self-attention

```python
def self_attention(input_sequence):
    """The input_sequence is a sequence of encoded vectors from
    a word embedding like space.  For example, for 600 words encoded
    in a 100d embedding, it would be shape (600, 100)

    Parameters
    ----------
    input_sequence : numpy array shape (max_tokens, dimensions)
        A single sequence, like a sentence for text, but encoded in a word embedding like space. Each
        sample is the vector of imbeddings for a single token.

    Returns
    -------
    output : numpy array shape (max_tokens, dimensions)
        Same shape array of token vectors returned, but all of the token vectors has been weighted and
        summed by attention, so each is now a context-aware token vector.
    """
    output = np.zeros(shape=input_sequence.shape)
    # iterate over each individual token from 0..max_tokens-1
    for i, pivot_vector in enumerate(input_sequence):

        # step 1 compute relevancy scores e.g. attention 
        # compute the attention score, which is dot product between the token
        # and every other token in this input sequence
        scores = np.zeros(shape=(len(input_sequence),))
        for j, vector in enumerate(input_sequence):
            scores[j] = np.dot(pivot_vector, vector.T)

        # scale by a normalization factor and apply softmax, result is probability
        # distribution that sums up to 1, but still basically importance scores
        scores /= np.sqrt(input_sequence.shape[1])
        scores = softmax(scores)

        # now step 2, compute sum of all word vectors, weighted
        # by relevancy, resulting in new representation of this token
        new_pivot_representation = np.zeros(shape=pivot_vector.shape)
        for j, vector in enumerate(input_sequence):
            new_pivot_representation += vector * scores[j]

        # so on output, the vector representation for token i is the new 
        # context-aware representation
        output[i] = new_pivot_representation
    return output
```

Keras has a built-in layer to handle learning and using a self-attention embedding:

In [6]:
num_heads = 4
embed_dim = 256

inputs = keras.Input(shape=(64,), dtype="int64")
mha_layer = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
outputs = mha_layer(inputs, inputs, inputs)

If you look closely you might notice:

- Why are we passing the inputs to the layer *three* times?
- What are these "multiple heads" we're referring to?

#### Generalized self-attention: the query-key-value model

### 11.4.2 Multi-head Attention



## Summary

<font color='blue'>
    
- Word order in Text processing is handled in 2 basic ways:
  1. **bag-of-words models** : discard order and treat as an unordered set (multi-hot encoding typically, 20,000 sparse vectors).
  2. **sequence models**: process words in order they appear like a timeseries
- For sequence models, can encode sequence again using one-hot encoding, but this ends up with very large input to RNN models.
- **word embeddings** are vector representations of words that map humanlanguage into a structured geometric space.